In [4]:
from pyspark.sql import functions as F

StatementMeta(, 8eb7db6f-19e4-4854-aeed-cf600531f02b, 6, Finished, Available, Finished, False)

In [5]:
df = spark.read.table("silver_transactions")

print(df.count())
display(df.limit(5))

StatementMeta(, 8eb7db6f-19e4-4854-aeed-cf600531f02b, 7, Finished, Available, Finished, False)

6362620


SynapseWidget(Synapse.DataFrame, 56701416-cb57-42f0-bf40-ddcd7e1c7f20)

In [6]:
gold_summary = (
    df.groupBy(
        "transaction_type",
        "transaction_year",
        "transaction_month"
    )
    .agg(
        F.count("*").alias("transaction_count"),
        F.sum("amount").alias("total_amount"),
        F.avg("amount").alias("average_amount"),
        F.sum(F.col("is_fraud").cast("int")).alias("fraud_count")
    )
)

StatementMeta(, 8eb7db6f-19e4-4854-aeed-cf600531f02b, 8, Finished, Available, Finished, False)

In [9]:
display(gold_summary)

StatementMeta(, 8eb7db6f-19e4-4854-aeed-cf600531f02b, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e0059570-c5e7-4a08-9b5a-63ae93860d57)

In [10]:
gold_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_transaction_summary")

StatementMeta(, 8eb7db6f-19e4-4854-aeed-cf600531f02b, 12, Finished, Available, Finished, False)

In [12]:
df_branches = spark.read.table("silver_branches")

print(df_branches.count())
display(df_branches.limit(5))

StatementMeta(, 8eb7db6f-19e4-4854-aeed-cf600531f02b, 14, Finished, Available, Finished, False)

200


SynapseWidget(Synapse.DataFrame, ffca733f-fadb-470a-a7d1-0dd71c70cf5b)

In [17]:
gold_branch_summary = (
    df_branches
    .groupBy("region")
    .agg(
        F.count("*").alias("total_branches"),

        F.sum(
            F.when(F.col("is_active") == True, 1).otherwise(0)
        ).alias("active_branches"),

        F.sum(
            F.when(F.col("is_active") == False, 1).otherwise(0)
        ).alias("inactive_branches"),

        F.round(F.avg("staff_count"), 1).alias("avg_staff_count"),

        F.max("staff_count").alias("max_staff_count"),

        F.min("staff_count").alias("min_staff_count")
    )
)

StatementMeta(, 8eb7db6f-19e4-4854-aeed-cf600531f02b, 19, Finished, Available, Finished, False)

In [18]:
gold_branch_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_branch_summary")

StatementMeta(, 8eb7db6f-19e4-4854-aeed-cf600531f02b, 20, Finished, Available, Finished, False)

In [19]:
df_customers = spark.read.table("silver_customers")
display(df_customers)

StatementMeta(, 8eb7db6f-19e4-4854-aeed-cf600531f02b, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 65232794-1b93-42cb-a18a-66dd1a585a9f)

In [20]:
gold_customer_summary = (
    df_customers
    .groupBy("segment", "age_band")
    .agg(
        F.count("*").alias("customer_count"),
        F.round(F.avg("annual_income"),2).alias("avg_annual_income"),
        F.round(F.avg("credit_score"),2).alias("avg_credit_score"),
        F.sum(
            F.when(F.col("is_active") == True, 1).otherwise(0)
        ).alias("active_customers"),
        F.sum(
            F.when(F.col("is_active") == False, 1).otherwise(0)
        ).alias("inactive_customers")
    )
    .orderBy("segment", "age_band")
)
display(gold_customer_summary)

StatementMeta(, 8eb7db6f-19e4-4854-aeed-cf600531f02b, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7a7ac2d1-a309-4286-803e-e7abcc58cebf)

In [21]:
gold_customer_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_customer_summary")

StatementMeta(, 8eb7db6f-19e4-4854-aeed-cf600531f02b, 23, Finished, Available, Finished, False)

In [22]:
df_products=spark.read.table("silver_products")
display(df_products)

StatementMeta(, 8eb7db6f-19e4-4854-aeed-cf600531f02b, 24, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6a9d63cb-d5c6-4b65-8857-049198c87be4)

In [23]:
gold_product_summary = (
    df_products
    .groupBy("category")
    .agg(
        F.count("*").alias("total_products"),
        F.avg("fee").alias("avg_fee"),
        F.max("fee").alias("max_fee"),
        F.min("fee").alias("min_fee"),
        F.avg("interest_rate").alias("avg_interest_rate"),
        F.max("interest_rate").alias("max_interest_rate"),
        F.min("interest_rate").alias("min_interest_rate")
    )
    .orderBy("category")
)
display(gold_product_summary)

StatementMeta(, 8eb7db6f-19e4-4854-aeed-cf600531f02b, 25, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0ccef61a-b47d-49df-b440-b884b5d68260)

In [24]:
gold_product_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_product_summary")

StatementMeta(, 8eb7db6f-19e4-4854-aeed-cf600531f02b, 26, Finished, Available, Finished, False)